# NUTDTS 816 · Assignment 1: Exploratory analysis and decomposition (7%)

Due Monday 21 September 2026, 23:59 WAT. Series A: nigeria_cpi. Series B: bonny_light.

**Name:** Precious Faseyosan **Matric No.:** 252325005 **Date:** 20 September, 2026.

This notebook must run top to bottom from a fresh Colab runtime. Keep the section headings; put your commentary in the markdown cells and your code in the code cells. Finish with the AI-use statement.

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://github.com/PreciousFaseyosan/NUTDTS816-Time_Series_Analysis-_Course_Repository/raw/main/"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

^C
ERROR: Operation cancelled by user


## 1. Load and tidy (DatetimeIndex, frequency, gaps)

*tsdata loaders; asfreq; explain any gap handling*

Task: Load and tidy both series into pandas with a proper DatetimeIndex at the correct frequency. Identify and handle any missing or duplicated timestamps and explain your choice.

In [ ]:
# Task 1a - Series A: Nigeria CPI
raw = pd.read_csv('data/nigeria_cpi.csv')
raw['date'] = pd.to_datetime(raw['date'])
print('duplicated timestamps:', raw['date'].duplicated().sum())
print('empty values in file :', raw['cpi'].isna().sum())

cpi = raw.drop_duplicates('date').set_index('date')['cpi']    # drop duplicated timestamps, then use the dates as the DatetimeIndex
cpi = cpi.asfreq('MS')                       # impose a regular monthly grid; missing timestamps become NaN
print('missing timestamps:', cpi.isna().sum())

cpi = cpi.interpolate(method='linear')       # fill any gap by linear interpolation

print(' ')
print(type(cpi), cpi.index.freq)
print(cpi.head(6))
print(cpi.index[:3])

In [ ]:
# Task 1b - Series B: Bonny Light
raw = pd.read_csv('data/bonny_light.csv')
raw['date'] = pd.to_datetime(raw['date'])
print('duplicated timestamps:', raw['date'].duplicated().sum())
print('empty values in file :', raw['bonny_light'].isna().sum())

bonny = raw.drop_duplicates('date').set_index('date')['bonny_light']
bonny = bonny.asfreq('MS')
print('missing:', bonny.isna().sum())

bonny = bonny.interpolate(method='linear')

print('')
print(type(bonny), bonny.index.freq)
print(bonny.head(6))
print(bonny.index[:3])

**Commentary:**

Both the Nigeria CPI and Bonny Light series have a proper monthly DatetimeIndex with a frequency of MS (Month Start). There were no duplicated timestamps or missing values in either dataset.

I used drop_duplicates('date') to remove any duplicate timestamps if they were present. I then used asfreq('MS') to place each series on a regular monthly frequency. If any months had been missing, asfreq() would have created those timestamps with NaN values. The gaps would then be filled using linear interpolation with interpolate(method='linear').

Since the output shows zero missing values for both series, no interpolation was needed.


## 2. Time, seasonal, subseries and lag plots for each series

*one figure per plot type; annotate what each shows*

Task: For each series produce a time plot, a seasonal plot, a seasonal subseries plot (where a season exists) and a lag plot. Annotate what each plot reveals.

In [ ]:
# Task 2 - functions definition
def seasonal_plot(s, period_label='year', title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    piv = df.pivot(index='month', columns='year', values='v')
    ax = piv.plot(legend=False, colormap='viridis', title=title, figsize=(8, 3.4))
    ax.set_xlabel('Month'); ax.set_xticks(range(1, 13))
    last = piv[piv.columns[-1]].dropna()     # the latest year can be incomplete (Bonny Light stops in June 2026)
    ax.annotate(str(piv.columns[-1]), (last.index[-1], last.iloc[-1]), fontsize=8)
    return ax

def subseries_plot(s, title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    fig, axes = plt.subplots(1, 12, figsize=(10, 3), sharey=True)
    for m, ax in zip(range(1, 13), axes):
        sub = df[df.month == m]
        ax.plot(sub.year, sub.v, lw=1); ax.axhline(sub.v.mean(), color='#B8860B', lw=1.2)
        ax.set_title(['J','F','M','A','M','J','J','A','S','O','N','D'][m-1]); ax.set_xticks([]); ax.grid(False)
    fig.suptitle(title, y=1.02); return fig

def lag_plot_grid(s, lags=(1, 2, 3, 6, 12, 24), title=''):
    fig, axes = plt.subplots(2, 3, figsize=(9, 5.5))
    for k, ax in zip(lags, axes.flat):
        ax.scatter(s.shift(k), s, s=8); ax.set_title(f'lag {k}'); ax.set_xlabel(f'x(t-{k})'); ax.set_ylabel('x(t)')
    fig.suptitle(title)
    fig.tight_layout()
    return fig

print('User-defined functions complete.')

In [ ]:
# Task 2a - time plots, with key dates marked

print('Time Plots of Nigeria CPI and Bonny Light Spot Prices')
print(' ')
ax = cpi.plot(title='A. Nigeria CPI (2024 = 100), monthly'); ax.set_xlabel(''); ax.set_ylabel('index')
rebase = pd.Timestamp('2025-01-01')                     # the rebasing date documented in data/README.md
ax.annotate('Jan 2025: index rebased (data/README.md)', xy=(rebase, cpi[rebase]),
            xytext=(pd.Timestamp('2017-01-01'), 100), arrowprops=dict(arrowstyle='->'), fontsize=8)
plt.show()

print(' ')
ax = bonny.plot(title='B. Bonny Light spot price (USD/bbl), monthly'); ax.set_xlabel(''); ax.set_ylabel('USD per barrel')
low, high = bonny.idxmin(), bonny.idxmax()              # dates of the lowest and highest price
spike = pd.Timestamp('2026-03-01')                      # the one-month jump flagged in the data provenance note
ax.annotate(f'Lowest: {low:%b %Y} (${bonny[low]:.0f})', xy=(low, bonny[low]),
            xytext=(pd.Timestamp('2015-06-01'), 18), arrowprops=dict(arrowstyle='->'), fontsize=8)
ax.annotate(f'Highest: {high:%b %Y} (${bonny[high]:.0f})', xy=(high, bonny[high]),
            xytext=(pd.Timestamp('2017-01-01'), 118), arrowprops=dict(arrowstyle='->'), fontsize=8)
ax.annotate(f'Mar 2026: +{100 * bonny.pct_change()[spike]:.0f}% in one month', xy=(spike, bonny[spike]),
            xytext=(pd.Timestamp('2022-06-01'), 25), arrowprops=dict(arrowstyle='->'), fontsize=8)
plt.show()


**Commentary - Time Plots:**

- **Nigeria CPI**: The CPI series shows a clear and steady upward trend from 2014 to 2025. The series rises fairly smoothly over the period, although the rate of increase varied across years and became much higher in later years (specifically around 2023 to 2024, when the monthly changes in the Task 7 plot are highest). The one visible break is a small dip in January 2025 (marked on the plot), which the data README documents as the rebasing of the index, a change of method and not a fall in prices.

- **Bonny Light**: The Bonny Light series does not show a steady upward trend like the CPI series. Rather, it is highly volatile and had different highs and lows. Prices fell sharply around 2020, reaching a low of roughly 14 USD/bbl in April 2020, before rising to a peak of around 130 USD/bbl in June 2022. Prices then fluctuate mostly within a lower range between 2023 to 2025, before another sharp increase occurred in 2026.

In [ ]:
# Task 2b - seasonal plots

print('Seasonal Plots of Nigeria CPI and Bonny Light Spot Prices')
print('')
seasonal_plot(cpi,   title='A. CPI seasonal plot, one line per year (dark = early, light = late)')
plt.show()
print('')
seasonal_plot(bonny, title='B. Bonny Light seasonal plot, one line per year (dark = early, light = late)')
plt.show()


**Commentary - Seasonal Plots:**

- **Nigeria CPI**: The seasonal plot shows a strong upward movement across years, with later years consistently lying above earlier years. This mainly reflects the long-term upward trend in CPI and shows no clear evidence of a repeating seasonal pattern.
- **Bonny Light**: The Bonny Light series is quite noisy due to the large fluctuations of oil prices, making any repeating monthly pattern (seasonality) difficult to identify visually.

In [ ]:
# Task 2c - seasonal subseries plots

print('Seasonal Subseries Plots of Nigeria CPI and Bonny Light Spot Prices')
print('')
subseries_plot(cpi,   'A. CPI seasonal subseries plot. Horizontal bar = mean of that month')
plt.show()
print('')
subseries_plot(bonny, 'B. Bonny Light seasonal subseries plot. Horizontal bar = mean of that month')
plt.show()


**Commentary - Seasonal Subseries Plots:**

- **Nigeria CPI**: The seasonal subseries plot shows the CPI values for each calendar month across the years. The monthly means (orange horizontal bars) increase consistently from January to December. This is mainly because CPI has a strong upward time trend, meaning that later months generally correspond to later points in time and therefore higher CPI levels. The increase from January to December may not necessarily be interpreted as evidence of seasonality.
- **Bonny Light**: The monthly means in the seasonal subseries plot for Bonny Light do not show a consistent pattern across the months. This reflects the substantial variability in oil prices across the years, which makes it difficult to identify a stable repeating monthly seasonal pattern.

In [ ]:
# Task 2d - lag plots

print('Lag Plots of Nigeria CPI and Bonny Light Spot Prices')
print('')
lag_plot_grid(cpi,   title='A. CPI lag plots')
plt.show()
print('')
lag_plot_grid(bonny, title='B. Bonny Light lag plots')
plt.show()


**Commentary - Lag Plots:**

- **Nigeria CPI**: The lag plots show a very strong positive relationship between the current CPI value (x_t) and its previous values (x_{t-1}, x_{t-2}, ...). This indicates strong serial dependence and persistence in the series. The strong relationship is expected because CPI changes gradually over time and has a strong upward trend.

- **Bonny Light**: The lag plots show considerably more scatter than the CPI lag plots, reflecting the greater volatility of oil prices. At lag 1 the points still sit fairly close to a rising line, so this month's price is a reasonable guide to next month's, but by lags 12 and 24 the points are widely spread with little visible pattern. The dependence is therefore weaker than for CPI and fades as the lag lengthens.

## 3. ACF of each series and of its first difference

*plot_acf; interpret trend, seasonality, dependence*

Task: Plot the sample autocorrelation function of each series and of its first difference. Interpret what the ACF tells you about trend, seasonality and dependence.

In [ ]:
# Task 3 - ACF of each series and of its first difference
from statsmodels.graphics.tsaplots import plot_acf
d_cpi   = cpi.diff().dropna()
d_bonny = bonny.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(cpi,   lags=36, ax=axes[0], title='A. CPI level')
plot_acf(d_cpi, lags=36, ax=axes[1], title='A. CPI, first difference')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(bonny,   lags=36, ax=axes[0], title='B. Bonny Light level')
plot_acf(d_bonny, lags=36, ax=axes[1], title='B. Bonny Light, first difference')
plt.show()

# - Ljung-Box test
from statsmodels.stats.diagnostic import acorr_ljungbox
for name, s_ in [('CPI level', cpi), ('CPI first diff', d_cpi), ('Bonny level', bonny), ('Bonny first diff', d_bonny)]:
    q = acorr_ljungbox(s_, lags=[12], return_df=True)
    print(f'{name:16s} Q(12) = {q.lb_stat.iloc[0]:9.2f}   p-value = {q.lb_pvalue.iloc[0]:.4f}')


**Commentary:**

- **Nigeria CPI**: The CPI level is trend-dominated, with the ACF showing a slow fade and strong positive autocorrelation persisting across many lags. This reflects the strong upward trend in the series as nearby observations tend to have similar values. After first differencing, the ACF declined more quickly, showing that differencing reduced the influence of the trend. However, the differenced series still showed strong, slowly decaying persistence, meaning that monthly CPI changes remained related across several months. There are no clear spikes at lags 12, 24, or 36, so there is no strong evidence of annual seasonality.

- **Bonny Light**: The Bonny Light level also shows a slowly decaying ACF, indicating persistence in monthly oil prices, although the dependence weakens more quickly than for CPI. After first differencing, most ACF values fell within the confidence band, with only a few short-lag correlations remaining. This indicates that differencing substantially reduced the persistence in the series. There are no clear spikes at lags 12, 24, or 36, hence annual seasonality is not evident. The Ljung-Box test for the first difference gives Q(12) = 21.61 and p = 0.0421, indicating that some dependence remains across the first 12 lags despite the weaker individual autocorrelations.

## 4. Additive or multiplicative? Justify

*evidence from the plots and the log plot*

Task: Decide whether an additive or a multiplicative structure is more appropriate for each series and justify the decision with evidence from the plots.

In [ ]:
# Task 4 - original vs log scale

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
cpi.plot(ax=axes[0], title='A. CPI: original scale')
np.log(cpi).plot(ax=axes[1], title='A. CPI: log scale')
for ax in axes: ax.set_xlabel('')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
bonny.plot(ax=axes[0], title='B. Bonny Light: original scale')
np.log(bonny).plot(ax=axes[1], title='B. Bonny Light: log scale')
for ax in axes: ax.set_xlabel('')
plt.show()


**Commentary:**

The Nigerian CPI series is multiplicative. **Evidence**: On the original scale, the CPI curve bends upward, rising faster in later years, while on the log scale it is much closer to a straight line, consistent with growth at a broadly steady percentage rate. The log line still steepens around 2023 to 2024 and shows the small dip in January 2025, so it is not perfectly straight.

The Bonny Light series is also treated as multiplicative, but the evidence is weaker than for CPI. **Evidence**: On the original scale, the swings around the 2022 and 2026 peaks are larger in dollar terms than those at the lower price levels of 2015 to 2017, while on the log scale the swings at high and low price levels look more similar in size. However, the log scale does not make the swings constant: it makes the April 2020 collapse look far deeper than any other movement. The log is therefore a pragmatic choice, and the Box-Cox fit in Task 6 (λ = 0.66) points to a milder transformation than the log.

## 5. Classical and STL decomposition compared

*seasonal_decompose vs STL; where do they disagree and why*

Task: Apply a classical decomposition and an STL decomposition to each series. Compare the trend and seasonal components and comment on where they disagree and why.

In [ ]:
# Task 5a - classical decomposition

from statsmodels.tsa.seasonal import seasonal_decompose, STL
MODEL_A = 'multiplicative'
MODEL_B = 'multiplicative'

dec_a = seasonal_decompose(cpi,   model=MODEL_A, period=12)
dec_b = seasonal_decompose(bonny, model=MODEL_B, period=12)

fig, axes = plt.subplots(4, 2, figsize=(11, 8), sharex='col')
for col, (dec, name) in enumerate([(dec_a, f'A. CPI, classical ({MODEL_A})'), (dec_b, f'B. Bonny Light, classical ({MODEL_B})')]):
    for row, comp in enumerate(['observed', 'trend', 'seasonal', 'resid']):
        getattr(dec, comp).plot(ax=axes[row, col], lw=1); axes[row, col].set_ylabel(comp); axes[row, col].set_xlabel('')
    axes[0, col].set_title(name)
plt.show()


In [ ]:
# Task 5b - STL decomposition

y_a = np.log(cpi)   if MODEL_A == 'multiplicative' else cpi
y_b = np.log(bonny) if MODEL_B == 'multiplicative' else bonny
stl_fit_a = STL(y_a, period=12, seasonal=13, robust=True).fit()
stl_fit_b = STL(y_b, period=12, seasonal=13, robust=True).fit()

fig = stl_fit_a.plot(); fig.set_size_inches(9, 7); fig.suptitle('A. CPI, STL (log scale if multiplicative)', y=1.0); plt.show()
print('')
print('')
fig = stl_fit_b.plot(); fig.set_size_inches(9, 7); fig.suptitle('B. Bonny Light, STL (log scale if multiplicative)', y=1.0); plt.show()


In [ ]:
# Task 5c - compare classical and STL on the ORIGINAL scale (exponentiate STL if it was fitted to logs, as in Lab 3 exercise / Lab 4)
def stl_on_original_scale(fit, model):
    back = np.exp if model == 'multiplicative' else (lambda x: x)
    return pd.DataFrame({'trend': back(fit.trend), 'seasonal': back(fit.seasonal), 'resid': back(fit.resid)})

stl_a = stl_on_original_scale(stl_fit_a, MODEL_A)
stl_b = stl_on_original_scale(stl_fit_b, MODEL_B)

def compare_plot(s, dec, stl, name):
    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    s.plot(ax=axes[0], color='lightgray', label='observed')
    dec.trend.plot(ax=axes[0], label='classical'); stl['trend'].plot(ax=axes[0], label='STL (robust)')
    axes[0].set_title(f'{name}: trend'); axes[0].legend()
    dec.seasonal.plot(ax=axes[1], label='classical'); stl['seasonal'].plot(ax=axes[1], label='STL (robust)')
    axes[1].set_title(f'{name}: seasonal component'); axes[1].legend()
    dec.resid.plot(ax=axes[2], label='classical', lw=1); stl['resid'].plot(ax=axes[2], label='STL (robust)', lw=1)
    axes[2].set_title(f'{name}: remainder'); axes[2].legend()
    for ax in axes: ax.set_xlabel('')
    plt.tight_layout(); plt.show()
    print('')
    print('')

compare_plot(cpi,   dec_a, stl_a, 'A. CPI')
compare_plot(bonny, dec_b, stl_b, 'B. Bonny Light')


In [ ]:
# Where does each method have no trend estimate?
for name, dec, stl in [('A. CPI', dec_a, stl_a), ('B. Bonny Light', dec_b, stl_b)]:
    print(f'{name}: classical trend valid from {dec.trend.first_valid_index():%b %Y} to {dec.trend.last_valid_index():%b %Y} | STL covers {stl.index[0]:%b %Y} to {stl.index[-1]:%b %Y}')


In [ ]:
# Task 5 - what the Jan 2025 rebasing seam does to each CPI decomposition (multiplicative, original scale)
win = slice('2024-10-01', '2025-03-01')
tbl = pd.DataFrame({'observed': cpi, 'STL trend': stl_a['trend'], 'STL seasonal': stl_a['seasonal'], 'STL remainder': stl_a['resid'],
                    'classical trend': dec_a.trend, 'classical seasonal': dec_a.seasonal, 'classical remainder': dec_a.resid})[win]
print(tbl.round(3).to_string())
print('lowest classical remainder in the whole sample:', f'{dec_a.resid.idxmin():%b %Y}', round(dec_a.resid.min(), 3))

**Commentary**:

The classical and STL decompositions produced broadly similar trend components in the middle of both series but differed more at the ends and around periods of unusual movements. **For CPI**, the classical trend was valid only from July 2014 to June 2025, compared with January 2014 to December 2025 for STL. **For Bonny Light**, the classical trend covered July 2015 to December 2025, while STL covered January 2015 to June 2026. These differences demonstrate the **end-loss weakness** of classical decomposition.

The seasonal components also differ because classical decomposition imposed an identical seasonal pattern every year, whereas STL allowed the seasonal pattern to evolve. This is particularly visible for CPI from around 2022 onward, where the STL seasonal component showed increasing amplitude and larger deviations, while the classical seasonal component continued to repeat the same pattern throughout the sample. For Bonny Light, the STL seasonal component also varied considerably from year to year, especially around 2015 - 2020, whereas the classical component imposed the same monthly pattern across all years. These differences illustrate the **frozen-seasonality weakness** of classical decomposition.

The **lack-of-robustness weakness** is most evident for Bonny Light around the 2020 price collapse: the classical trend dropped sharply toward the extreme observation and then quickly reversed, whereas the robust STL trend remained smoother and less affected by the collapse. The remainders show the other side of this: in April 2020 the STL remainder dips further than the classical one (about 0.23 against 0.32, read from the remainder panel), because the robust fit keeps the trend from bending towards the collapse and so leaves the shock in the remainder, as designed.

**The January 2025 rebasing break (CPI).** The published index steps down 2.8% between December 2024 and January 2025, which the data README documents as the seam of the rebasing, not a real fall in prices. Neither decomposition lets the dip pull the trend down: the STL and classical trends both keep rising through it. The break shows up in the remainder instead. From October to December 2024 the index runs roughly 4% to 6% above the STL trend (remainder 1.040, 1.051, 1.060), and the dip then resets it to about 1.00 in January 2025. In the classical decomposition, the January 2025 remainder (0.982) is the lowest in the whole sample. The two methods also differ in the seasonal component. The classical method uses a fixed January factor, so it cannot absorb the break, whereas the flexible STL seasonal absorbs a small part of it: its January 2025 factor is 0.9924, against about 0.9959 in a rough check with the seam smoothed out, roughly a third of a percentage point of the 2.8% drop.


## 6. Transformation decision

*log / Box-Cox; show the effect*

Task: Decide whether a transformation (log or Box–Cox) is warranted; if so apply it and show the effect on the decomposition.

In [ ]:
# Task 6a - fitted Box-Cox lambda, and the series on three scales (Lab 4)
from scipy import stats
lam_cpi   = stats.boxcox_normmax(cpi.values,   method='mle')
lam_bonny = stats.boxcox_normmax(bonny.values, method='mle')
print(f'MLE Box-Cox lambda: CPI = {lam_cpi:.3f} | Bonny Light = {lam_bonny:.3f}  (0 = log)')

for tag, s, lam in [('A. CPI', cpi, lam_cpi), ('B. Bonny Light', bonny, lam_bonny)]:
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))
    s.plot(ax=axes[0], title=f'{tag}: original (λ = 1)')
    np.log(s).plot(ax=axes[1], title=f'{tag}: log (λ = 0)')
    pd.Series(stats.boxcox(s.values, lmbda=lam), index=s.index).plot(ax=axes[2], title=f'{tag}: Box-Cox (λ = {lam:.2f})')
    for ax in axes: ax.set_xlabel('')
    plt.show()
    print('')


In [ ]:
# Task 6b - effect of the log on the STL decomposition: original scale vs log scale

for tag, s in [('A. CPI', cpi), ('B. Bonny Light', bonny)]:
    stl_orig = STL(s,         period=12, seasonal=13, robust=True).fit()
    stl_log  = STL(np.log(s), period=12, seasonal=13, robust=True).fit()
    fig, axes = plt.subplots(2, 2, figsize=(11, 5), sharex=True)
    stl_orig.seasonal.plot(ax=axes[0, 0], lw=1, title=f'{tag}: STL seasonal, original scale')
    stl_log.seasonal.plot(ax=axes[0, 1],  lw=1, title=f'{tag}: STL seasonal, log scale')
    stl_orig.resid.plot(ax=axes[1, 0],    lw=1, title=f'{tag}: STL remainder, original scale')
    stl_log.resid.plot(ax=axes[1, 1],     lw=1, title=f'{tag}: STL remainder, log scale')
    for ax in axes.flat: ax.set_xlabel('')
    plt.tight_layout(); plt.show()
    print('')

# Decisions
LOG_CPI   = True
LOG_BONNY = True


**Commentary**:

- **Fitted λ and the choice of the log.** The fitted Box-Cox λ is -0.43 for Nigeria CPI and 0.66 for Bonny Light. Rounded to simple, interpretable values, these are about -0.5 (inverse square root) and 0.5 (square root) respectively. Neither is close to 0, so the fitted values do not point directly to the log. I kept the log for both series as a pragmatic choice rather than one the fit demands: the course guidance is to round λ to an interpretable value and prefer the log unless there is a clear reason not to, the log matches the multiplicative structure identified in Task 4, and it is easy to interpret in percentage terms. The argument that a λ near 0 means the log gives up little applies only loosely here.

- **Nigeria CPI.** In the three-panel plot, the log and Box-Cox panels look broadly similar: both turn the upward-bending original curve into a much straighter line. The log line still steepens around 2023 to 2024, while the Box-Cox line bends the other way and flattens towards the end, so the log's shape is not clearly worse than Box-Cox. In the seasonal panels, the swings on the original scale grow steadily over time and are several times larger after about 2022 than in the early years. On the log scale the growth is much smaller, although the swings still widen after about 2022. The two scales use different units, so I compare how the swings change over time, not their sizes. In the remainder panels, the January 2025 rebasing appears as a build-up through late 2024 followed by a drop to about zero in January 2025, on both scales, so the log does not remove it. On the original scale the remainder is close to zero in the early years and much larger later, while on the log scale it is more even across the sample.

- **Bonny Light.** Its fitted λ of 0.66 is closer to 1 (no transformation) than to 0 (log), which makes the log the weaker choice for this series. In the three-panel plot, the Box-Cox panel keeps a shape closer to the original, while the log panel stretches the low end and turns the April 2020 collapse into a very deep spike. Unlike CPI, the seasonal swings do not grow over time on either scale: they are largest in the first few years and smaller afterwards. In the remainder panels, the original scale shows a fall of about 40 in April 2020 and an even larger rise of about 60 in 2026. On the log scale the April 2020 collapse dominates (roughly -1.5, more than twice the size of the 2026 rise), so the log makes the 2020 collapse look much bigger relative to everything else.

- **Decision.** I kept the log for both series for consistency and interpretability. It is reasonable for CPI. For Bonny Light the fitted λ points to a milder transformation, so the evidence does not strongly favour the log, and results that depend on it, especially around 2020, should be read with that in mind.


## 7. Seasonally adjusted series

*adjust on the right scale*

Task: Produce and plot the seasonally adjusted version of each series.

In [ ]:
# Task 7 - seasonal adjustment and plot
# multiplicative: divide by the seasonal factor; additive: subtract
def seasonally_adjust(s, stl_df, model):
    return s / stl_df['seasonal'] if model == 'multiplicative' else s - stl_df['seasonal']

cpi_sa   = seasonally_adjust(cpi,   stl_a, MODEL_A)
bonny_sa = seasonally_adjust(bonny, stl_b, MODEL_B)

for tag, s, sa, unit in [('A. CPI', cpi, cpi_sa, 'index'), ('B. Bonny Light', bonny, bonny_sa, 'USD/bbl')]:
    ax = s.plot(figsize=(9, 3.4), lw=1, label='observed')
    sa.plot(ax=ax, lw=1.5, color='#B8860B', label='seasonally adjusted')
    ax.set_title(f'{tag}: observed and seasonally adjusted'); ax.set_ylabel(unit); ax.legend(); ax.set_xlabel(''); plt.show()
    print('')


In [ ]:
# Task 7 - seasonally adjusted series on monthly % log change, raw vs adjusted
for tag, s, sa in [('A. CPI', cpi, cpi_sa), ('B. Bonny Light', bonny, bonny_sa)]:
    raw_chg, sa_chg = 100 * np.log(s).diff(), 100 * np.log(sa).diff()
    ax = raw_chg.plot(figsize=(10, 3.4), lw=1, label='raw')
    sa_chg.plot(ax=ax, lw=1.3, label='seasonally adjusted')
    ax.set_title(f'{tag}: monthly % log change, raw vs seasonally adjusted'); ax.legend(); ax.set_xlabel(''); plt.show()
    print('')

print('')
print('A. CPI seasonal factors, last 12 months (' + MODEL_A + '):');          print(stl_a['seasonal'].iloc[-12:].round(4).to_string())
print('\nB. Bonny Light seasonal factors, last 12 months (' + MODEL_B + '):'); print(stl_b['seasonal'].iloc[-12:].round(4).to_string())


In [ ]:
# Task 7 - numbers quoted in the Task 7 and Task 8 commentary
for tag, s, sa, stl in [('A. CPI', cpi, cpi_sa, stl_a), ('B. Bonny Light', bonny, bonny_sa, stl_b)]:
    raw_chg = 100 * np.log(s).diff()                     # monthly % log change, raw
    gap = (raw_chg - 100 * np.log(sa).diff()).abs()      # size of the raw-vs-adjusted gap each month
    print(tag)
    print('   seasonal factors, full sample:', round(stl['seasonal'].min(), 2), 'to', round(stl['seasonal'].max(), 2))
    print('   median gap (raw vs adjusted):', round(gap.median(), 2), '| median size of a monthly change:', round(raw_chg.abs().median(), 2))
    print('   months with the largest gaps:', [d.strftime('%b %Y') for d in gap.sort_values(ascending=False).index[:4]])


In [ ]:
# Task 7 - the Jan 2025 seam: raw vs seasonally adjusted monthly change, and the STL January factor
jan = pd.Timestamp('2025-01-01')
print('CPI monthly % log change in Jan 2025: raw', round(100 * np.log(cpi).diff()[jan], 1), '| seasonally adjusted', round(100 * np.log(cpi_sa).diff()[jan], 1))
print('STL seasonal factor, Dec 2024 -> Jan 2025:', round(float(stl_a['seasonal']['2024-12-01']), 4), '->', round(float(stl_a['seasonal'][jan]), 4))
print('STL January factor by year:', {y: round(float(stl_a['seasonal'][f'{y}-01-01']), 4) for y in range(2019, 2026)})

# Rough sensitivity check: rescale 2025 onward so the series is continuous across the seam, then refit the same STL
cpi_c = cpi.copy(); cpi_c[cpi_c.index >= jan] *= cpi['2024-12-01'] / cpi[jan]
stl_c = STL(np.log(cpi_c), period=12, seasonal=13, robust=True).fit()
print('Jan 2025 seasonal factor: as fitted', round(float(stl_a['seasonal'][jan]), 4), '| with the seam smoothed out', round(float(np.exp(stl_c.seasonal[jan])), 4))

**Commentary:**

The seasonal adjustment was performed on the log scale because the series were identified as multiplicative in Task 4. The estimated seasonal component was subtracted from the log-transformed series, and the result was exponentiated to return to the original scale. This removes the estimated seasonal effect while preserving the underlying trend and other non-seasonal movements.

For Nigeria CPI, the observed and seasonally adjusted level series are very close throughout the sample. The seasonal factors for the final 12 months ranged from 0.9787 to 1.0131, corresponding to adjustments of roughly ±2% relative to the observed level. This is small compared with the overall CPI level and its strong upward trend. In the monthly log-change plot, the raw and adjusted series are also very close, with a median gap of about 0.15 percentage points compared with a typical monthly change of about 1.11%. The larger divergence around January 2025 coincides with the rebasing break in the index (see Task 5). The STL seasonal factor moves from about 0.983 in December 2024 to 0.992 in January 2025, which pushes the adjusted January change to about -3.9% against -2.9% raw. The adjusted figure for that month therefore exaggerates the break and should not be read as a real fall in prices. The seasonal-strength measure in Task 8 was 0.000, providing no strong evidence of a substantial seasonal pattern.

For Bonny Light, the observed and seasonally adjusted level series are also generally close, although the adjustment is more visible than for CPI. The seasonal factors for the final 12 months ranged from 0.9070 to 1.0641, while the full-sample factors ranged from about 0.74 to 1.12. In the monthly log-change plot, the largest differences occur around January 2016, January 2017, February 2015, and January 2018, where the seasonal factor changes sharply between adjacent months. For monthly changes, the gap is therefore determined by the change in the seasonal factor between consecutive months, rather than simply by how far an individual factor is from 1. The median gap is about 3.4 percentage points compared with a typical monthly change of about 6.9%, so the seasonal adjustment can represent a meaningful part of an ordinary monthly movement even though it is small relative to major price shocks. The seasonal-strength measure of 0.000 in Task 8 nevertheless indicates no strong evidence of a stable seasonal pattern.

A manager would use the seasonally adjusted series when the objective is to assess underlying movements after removing estimated seasonal effects. For CPI, the adjustment has little effect relative to the overall level and long-term trend. For Bonny Light, it can have a more noticeable effect relative to a typical monthly price change, but it does not remove irregular shocks such as the 2020 price collapse or the large movements in 2022 and 2026.

## 8. Strength of trend and seasonality

*compute for both series and compare*

Task: Compute the strength-of-trend and strength-of-seasonality features for both series and use them to compare the two.

In [ ]:
def strength_features(s, period, log=False, seasonal=13):
    """Lecture 4 / Lab 4 definitions: F = max(0, 1 - Var(R) / Var(component + R)), from a robust STL fit."""
    y = np.log(s) if log else s
    r = STL(y, period=period, seasonal=seasonal, robust=True).fit()
    ft = float(max(0, 1 - r.resid.var() / (r.trend + r.resid).var()))
    fs = float(max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var()))
    return round(ft, 3), round(fs, 3)

# A white-noise series as a yardstick: by construction it has no trend and no seasonality
wn = pd.Series(np.random.default_rng(3).normal(size=144), index=cpi.index)

rows = []
for name, s, lg in [("A. Nigeria CPI", cpi, LOG_CPI), ("B. Bonny Light", bonny, LOG_BONNY), ("White noise (yardstick)", wn, False)]:
    ft, fs = strength_features(s, 12, log=lg); rows.append((name, lg, ft, fs))
print(pd.DataFrame(rows, columns=["series", "log used", "trend strength", "seasonal strength"]).to_string(index=False))

# Sensitivity: how much do the seasonal-strength numbers depend on the STL seasonal window?
print("\nSeasonal strength for different STL seasonal windows (odd numbers; small = the pattern may change quickly):")
for name, s, lg in [("A. Nigeria CPI", cpi, LOG_CPI), ("B. Bonny Light", bonny, LOG_BONNY)]:
    print(f"   {name:16s}", {w: strength_features(s, 12, log=lg, seasonal=w)[1] for w in (7, 13, 25)})


**Commentary:**

The strength-of-trend and strength-of-seasonality measures were computed from a robust STL fit on the log scale, using a seasonal period of 12 and a seasonal window of 13. The measures are unit-free, making them comparable across the two series. Trend strength is calculated as 1 - Var(R) / Var(T + R), while seasonal strength is 1 - Var(R) / Var(S + R). Values near 1 indicate a strong component, while values near 0 indicate that the component does not stand out from the remainder.

Nigeria CPI has a trend strength of 1.000, indicating a very strong slow-moving component, while Bonny Light has a lower value of 0.603. This agrees with Tasks 2 and 3: CPI was strongly trend-dominated, whereas Bonny Light was more volatile and showed slower swings between periods such as the 2016 low and 2022 peak. Thus, the 0.603 value for Bonny Light indicates substantial slow-moving structure rather than a steady directional trend. Both series have a seasonal strength of 0.000, consistent with the absence of clear repeating annual patterns in the seasonal plots and ACFs.

The seasonal-strength window check gives 0.000 for CPI across all three windows and 0.053, 0.000, and 0.000 for Bonny Light with windows 7, 13, and 25 respectively. This suggests that allowing the seasonal component to change more rapidly captures only a small amount of additional variation for Bonny Light.

The single white-noise yardstick series gives a trend strength of 0.045 and seasonal strength of 0.128, illustrating the levels that can arise from noise alone. The seasonal strength of 0.000 for both series therefore provides no evidence of a meaningful seasonal component. The much higher trend-strength values indicate that both series contain more slow-moving structure than this noise example, although the measure does not by itself establish a steady directional trend.

Bonny Light's seasonal factors can still deviate noticeably from 1 even with a seasonal strength of 0.000. Over the full sample, the factors range from about 0.74 to 1.12, but these deviations do not stand out as a stable repeating seasonal pattern. This illustrates why the strength measure is more informative than individual seasonal factors when assessing whether seasonality is systematic and substantial.

## 9. Narratives (one page per series, also in the PDF report)

*what the series is doing; regularities; events or breaks; what a forecaster should worry about*

Task: Write a one-page narrative for each series (two pages in total) addressed to a manager: what the series is doing, what is regular and what is not, what events or breaks are visible, and what a forecaster should be careful about.

In [ ]:
# Task 9 - numbers quoted in the two narratives
chg = 100 * np.log(cpi).diff()                          # CPI monthly % log change
print('A. CPI')
print('   Jan 2014 -> Dec 2025:', round(cpi.iloc[0], 1), '->', round(cpi.iloc[-1], 1), '| times higher:', round(cpi.iloc[-1] / cpi.iloc[0], 1))
print('   average monthly % change in 2014, 2023, 2024:', [round(float(chg[str(y)].mean()), 2) for y in (2014, 2023, 2024)])
print('   largest monthly rises in 2023-2024:', [f'{d:%b %Y} ({v:.1f}%)' for d, v in chg['2023':'2024'].nlargest(3).items()])
print('   months with a fall (% log change):', [f'{d:%b %Y} ({v:.1f}%)' for d, v in chg[chg < 0].items()])
print('   Dec 2024 -> Jan 2025:', round(cpi['2024-12-01'], 2), '->', round(cpi['2025-01-01'], 2), f"({100 * (cpi['2025-01-01'] / cpi['2024-12-01'] - 1):.1f}%)")
print('B. Bonny Light')
print('   Jan 2015:', round(bonny.iloc[0], 2), '| Jan 2020:', round(bonny['2020-01-01'], 2), '| Apr 2020 low:', round(bonny['2020-04-01'], 2), '| Jun 2020:', round(bonny['2020-06-01'], 2))
print('   Jun 2022 high:', round(bonny['2022-06-01'], 2))
print('   2023-2025 range:', round(bonny['2023':'2025'].min(), 1), 'to', round(bonny['2023':'2025'].max(), 1))
print('   Feb, Mar, Apr, Jun 2026:', [round(float(bonny[d]), 2) for d in ('2026-02-01', '2026-03-01', '2026-04-01', '2026-06-01')], '| Mar 2026 change: {:.0f}%'.format(100 * bonny.pct_change()['2026-03-01']))

Page 1:

**Series A: Nigeria Consumer Price Index (CPI)**

**Bottom line.** Expect prices to keep rising, but do not assume the recent pace will hold: the index has climbed steadily for twelve years, to more than six times its January 2014 level, and rose fastest in 2023 and 2024. The one-month dip in January 2025 is a change in how the index is measured, not falling prices, so it should not be read as a turn.

The index rose from about 20 in January 2014 to about 131 in December 2025 (2024 = 100). Growth was gentle at first, averaging 0.6% a month in 2014, but quickened to 2.1% a month in 2023 and 2.5% in 2024, with single months near 3%. Across the whole period the typical monthly rise is about 1.1%, so 2023 and 2024 ran at about twice the usual pace.

Nothing repeats reliably by calendar month. Average levels do rise from January to December, but that is only the long upward run, and a formal check found no dependable monthly pattern; at most, a seasonal adjustment would move the index by about 2%. I am confident of this. What is dependable is momentum: the index typically moves only about 1.1% from one month to the next, so this month is a very strong guide to next month.

One break and one change of pace stand out. In January 2025 the index fell 2.8% in a single month (113.9 to 110.7). The data note attributes this to the switch to a new base year, not to a real fall in prices. The 2023 to 2024 speed-up is visible in the data but its cause is not; my own interpretation, which I have not tested, is that the 2023 removal of the fuel subsidy and the naira devaluation contributed.

A forecaster should be careful about three things. **The January 2025 break.**  A forecast that treats the dip as real will misjudge the trend, so adjust for it explicitly or compare growth rates rather than raw levels across it. **The changing pace.**  Monthly rises have ranged from under 1% to about 3%, so the past pace is a poor guide to the future; base forecasts on recent months and show a range, not a single growth rate. **Easy-looking levels.**  Because this month is such a strong guide to next month, even a lazy forecast of the level looks accurate, so judge forecasts on how well they predict the monthly change, not the level.


Page 2:

**Series B: Bonny Light crude oil price (USD per barrel)**

**Bottom line.** Plan around ranges and scenarios rather than a single forecast, and do not expect a calendar rhythm to help. Bonny Light has no steady direction; it moves in long swings, with sudden shocks such as the collapse to $14 a barrel in April 2020 and a 47% one-month jump in March 2026.

Prices started at about $49 a barrel in January 2015 and moved in long swings rather than one direction: down to $14 in April 2020, up to a peak of $130 in June 2022, then within a band of roughly $64 to $98 through 2023 to 2025. In 2026 the price surged again, reaching about $127 in April before easing to $88 in June, the last month of data.

There is no dependable calendar pattern: only the most flexible check found a trace of one, and it was tiny, so I would not use the month of the year as a guide. This month says something about next month, but far less than for CPI, and the link fades within a year. The typical month-to-month move is about 7%, roughly six times CPI's.

Three episodes stand out. In April 2020 the price fell from $67 in January to $14, then recovered to $40 by June; the data note calls these COVID-era swings, and weak pandemic demand is the likely cause (my interpretation). The June 2022 peak of $130 I interpret as the global energy-price spike after Russia's invasion of Ukraine. In March 2026 the price jumped 47% in one month ($72 to $106); the data note flags it, and I have no tested explanation.

A forecaster should be careful about three things. **Sudden shocks.**  A fall to $14, a rise to $130 and a 47% one-month jump can each dominate a forecast, so give ranges and high/low scenarios rather than one number. **Unusual months.**  A few extreme months, especially April 2020, distort averages and estimates of normal behaviour, so check results with and without them and do not rely on a fixed calendar adjustment. **Old behaviour.**  The price has moved between very different levels, and the 2026 surge has already partly reversed ($127 in April, $88 in June), so weight recent months more heavily and refresh the forecast as new data arrive.


## AI-use statement

(Two to five lines: which tools, for what. 'No AI tools were used' is acceptable.)

I used Claude (Anthropic) for explanations of concepts, for writing and debugging code, and for feedback on my commentaries, and ChatGPT (OpenAI) to rephrase first drafts of my commentaries.

In the final stage I used Claude to apply quality-control review comments to my commentaries (including rewriting Task 6 after I changed and reran its code), to add a cell that prints the numbers quoted in the narratives, and to draft the Task 9 narratives and this report from my commentaries.

The analysis and code are my own. The narrative wording was drafted by Claude from my commentaries; I reviewed it and take responsibility for it.
